# 52a · XGB EHR-only **out-of-fold** — nguyên liệu cho phép thử "bộ mã hoá có phải nút thắt"

## Câu hỏi

`46` đo: `tok 0.1164` vs `XGB 0.1241` ⇒ bộ mã hoá neural **kém `−0.0077`**. Suy tỉ lệ thì khoảng
cách đó chuyển sang fusion chỉ còn `~+0.0005` macro — **dưới sàn nhiễu `0.0007`**. Nhưng đó là **suy
luận**, không phải phép đo.

Phép đo dứt điểm: **cho fusion ăn thẳng đầu ra của XGB**, rồi so `G_ehr` của hai đường.

```
G_ehr(xgb-feat) >> G_ehr(ColTokBag)  ->  bo ma hoa DUNG la nut that, dang sua
~ bang nhau                          ->  KHONG phai; tran nam o NGUON NHAN
                                         (nhan CXR-LT trich tu chinh report cua phim do)
```

Notebook này chỉ làm **nguyên liệu**: ma trận `[377,110 × 40]` dự đoán XGB EHR-only.
`52b` mới là lần chạy fusion.

## Vì sao phải OUT-OF-FOLD, không dùng thẳng model của `46`

`46` huấn luyện XGB trên **toàn bộ** lát khớp `TF_` rồi dự đoán `VA`. Nếu lấy đúng model đó dự đoán
**chính `TF_`** thì mỗi hàng train sẽ nhận một dự đoán **đã thấy nhãn của chính nó** ⇒ đặc trưng gần
như là bản sao của nhãn ⇒ đầu đọc fusion học tắt qua nó, và `G_ehr(xgb-feat)` sẽ **khổng lồ mà vô
nghĩa**.

```
TF_   ->  5 fold theo SUBJECT. Moi hang nhan du doan cua model KHONG thay subject cua no.
SL/VA ->  model huan luyen tren TOAN BO TF_. Sach vi TF_ / SL / VA roi nhau theo subject.
test  ->  KHONG du doan; dien hang so prevalence. Ky luat "test CHUA HE cham" giu nguyen.
```

Chia fold **theo subject**, không theo hàng: một bệnh nhân trung bình 5.8 ảnh, chia theo hàng thì
ảnh khác của cùng người rơi vào fold train ⇒ rò rỉ.

## Đặc trưng — **verbatim `46`**

`XG = [X (217 cột) ; XB (304 đếm vi khuẩn)] = 521`. Cùng `29b`+`29c`, cùng z-score bằng thống kê
`TF_`, cùng bỏ 30 cột one-hot `micro_org_*`. Siêu tham số XGB **y hệt** `46`.

## C0 · Config + lát cắt — cùng cách cắt như `35`/`37`/`38*`/`44*`/`45`/`46`

In [2]:
import os, json, math, time, re
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES52   = os.path.join(OUT_DIR, 'stage4_nb52a')
def ensure_res():
    os.makedirs(RES52, exist_ok=True); return RES52
ensure_res()
json.dump({'notebook': '52a'}, open(os.path.join(RES52, '_run_info.json'), 'w'))

SEED = 42
K_FOLD = 5
XGB_ROUNDS, XGB_DEPTH = 300, 6     # y het 46
MOC46_VA = 0.1241                  # so XGB cua 46 tren val — C3 phai TAI LAP duoc
EHR_CLIP = 10.0

P_OOF  = os.path.join(RES52, 'xgb_oof.npy')
P_PROG = os.path.join(RES52, '_oof_progress.json')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
TEST = ~(TR | VA)

# --- chia fold THEO SUBJECT trong TF_ ---
_su = np.unique(SUB[TF_]); _rsf = np.random.default_rng(SEED + 1); _rsf.shuffle(_su)
FOLD_OF_SUB = {int(s): int(k % K_FOLD) for k, s in enumerate(_su)}
FOLD = np.full(len(ORDER), -1, 'int64')
FOLD[TF_] = [FOLD_OF_SUB[int(s)] for s in SUB[TF_]]
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,} | '
      f'test {TEST.sum():,}')
print(f'{K_FOLD} fold theo subject tren lat khop: ' +
      ' '.join(f'f{k}:{(FOLD==k).sum():,}' for k in range(K_FOLD)))
# --- cong 1: MOI subject nam o DUNG MOT fold. Day la cong CHINH XAC TUYET DOI, khong nguong.
#     Chia fold theo HANG se cho `nunique > 1` ngay lap tuc: mot benh nhan trung binh 5.8 anh nen
#     anh khac cua chinh no lot sang fold train => model da thay benh nhan do => ro ri.
#     (Cong thong ke "mAP(OOF) co cao hon val khong" KHONG thay the duoc: no tron ro ri voi viec
#      model OOF chi hoc tren 80% du lieu, nen dry-run cho thay no khong bat duoc.) ---
_nf = pd.Series(FOLD[TF_]).groupby(SUB[TF_]).nunique()
assert int(_nf.max()) == 1, (f'{int((_nf > 1).sum()):,} subject nam o NHIEU fold -> dang chia fold '
                             'theo HANG chu khong theo SUBJECT. Ro ri chac chan.')
for k in range(K_FOLD):
    a = set(SUB[TF_ & (FOLD == k)]); b = set(SUB[TF_ & (FOLD != k)])
    assert not (a & b), f'fold {k}: {len(a & b)} subject nam ca hai ben -> ro ri'
print(f'cong 1: moi subject o DUNG mot fold | {K_FOLD} fold roi nhau theo subject OK')

Mounted at /content/drive
377,110 anh | khop 219,785 | chon 39,086 | val 39,293 | test 78,946
5 fold theo subject tren lat khop: f0:44,345 f1:43,127 f2:43,703 f3:44,417 f4:44,193
cong 1: moi subject o DUNG mot fold | 5 fold roi nhau theo subject OK


## C1 · Đặc trưng cho XGB — **verbatim `46` C1**

In [3]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()

def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = {'Q_FULL': [c for c in SPEC['Q_FULL'] if c in E_DF.columns]}
EHR  = {k: _mat(v) for k, v in COLS.items()}
for k, m in EHR.items():
    # Cot NHI PHAN thi KHONG z-score (46/45: p = 1/219,785 -> gia tri chuan hoa 468).
    b_ = np.array([np.isin(np.unique(m[:, j]), (0.0, 1.0)).all() for j in range(m.shape[1])])
    mu = m[TF_].mean(0, keepdims=True); sd = m[TF_].std(0, keepdims=True) + 1e-6
    mu[:, b_] = 0.0; sd[:, b_] = 1.0
    EHR[k] = np.clip((m - mu) / sd, -EHR_CLIP, EHR_CLIP).astype('float32')
    print(f'  {k}: {int(b_.sum())}/{len(b_)} cot NHI PHAN (giu 0/1)')

_MICRO_RE = re.compile(r'^micro_org_')
_C0 = list(COLS['Q_FULL'])
_KEEP = [i for i, c in enumerate(_C0) if not _MICRO_RE.match(c)]
assert len(_C0) - len(_KEEP) == 30, f'bo {len(_C0)-len(_KEEP)} cot micro_org_*, phai la 30'
COLS['Q_FULL'] = [_C0[i] for i in _KEEP]; EHR['Q_FULL'] = EHR['Q_FULL'][:, _KEEP]
X = EHR['Q_FULL']
print(f'bo 30 cot one-hot micro_org_* (danh tinh di qua tui dem)')

P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
assert os.path.exists(P_MICRO), f'khong thay {P_MICRO} -> chay 29c truoc'
_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
V_MIC = int(json.load(open(P_VOCAB))['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max()))
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
XB = np.zeros((len(ORDER), V_MIC - 1), 'float32')      # tui GOP thanh dem — dung dang 46 cho XGB
for r in range(len(ORDER)):
    m = MCNT[r] > 0
    if m.any(): np.add.at(XB[r], MIDS[r][m], MCNT[r][m])
XG = np.concatenate([X, XB], 1)
# --- cong 2: hinh dang PHAI khop 46 (217 + 304 = 521) ---
assert XG.shape == (len(ORDER), X.shape[1] + V_MIC - 1), f'XG {XG.shape}'
assert np.isfinite(XG).all(), 'XG co gia tri khong huu han'
print(f'cong 2: XG {XG.shape} = {X.shape[1]} cot + {V_MIC-1} dem vi khuan (khop 46)')

  Q_FULL: 58/247 cot NHI PHAN (giu 0/1)
bo 30 cot one-hot micro_org_* (danh tinh di qua tui dem)
cong 2: XG (377110, 522) = 217 cot + 305 dem vi khuan (khop 46)


## C2 · XGB out-of-fold — `40 × (5 fold + 1 full)` = **240** lần fit

Resume theo **từng cặp `(nhãn, fold)`**: đứt thì mất đúng một lần fit.
`fold = -1` nghĩa là model huấn luyện trên **toàn bộ** `TF_`, dùng cho `SL` và `VA`.

Nhãn có `< 5` ca dương ở lát khớp: XGB không học nổi, điền **hằng số prevalence** (đúng cách `46`).

In [4]:
import xgboost as xgb
try:
    import torch; _gpu = torch.cuda.is_available()
except Exception:
    _gpu = False
XDEV = 'cuda' if _gpu else 'cpu'
PARAMS = dict(objective='binary:logistic', eval_metric='aucpr', max_depth=XGB_DEPTH,
              eta=0.1, subsample=0.8, colsample_bytree=0.8, tree_method='hist',
              device=XDEV, nthread=os.cpu_count() or 4)
print(f'XGB tren {XDEV} | {XGB_ROUNDS} vong | depth {XGB_DEPTH} | {C} nhan x ({K_FOLD}+1) = '
      f'{C*(K_FOLD+1)} lan fit')

OOF = (np.load(P_OOF) if os.path.exists(P_OOF)
       else np.full((len(ORDER), C), np.nan, 'float32'))
done = set(json.load(open(P_PROG))) if os.path.exists(P_PROG) else set()

I_TF = np.where(TF_)[0]
D_SL = xgb.DMatrix(XG[SL]); D_VA = xgb.DMatrix(XG[VA])
t0 = time.time()
for j in range(C):
    npos = int(YM[TF_, j].sum())
    if npos < 5:                       # khong hoc noi -> hang so prevalence (dung cach 46)
        OOF[TR | VA, j] = float(npos) / max(TF_.sum(), 1)
        done |= {f'{j}_{k}' for k in list(range(K_FOLD)) + [-1]}
        continue
    for k in list(range(K_FOLD)) + [-1]:
        key = f'{j}_{k}'
        if key in done: continue
        if k >= 0:
            m_tr = TF_ & (FOLD != k); m_pr = TF_ & (FOLD == k)
        else:
            m_tr = TF_; m_pr = None
        bst = xgb.train(PARAMS, xgb.DMatrix(XG[m_tr], label=YM[m_tr, j]),
                        num_boost_round=XGB_ROUNDS, verbose_eval=False)
        if k >= 0:
            OOF[m_pr, j] = bst.predict(xgb.DMatrix(XG[m_pr]))
        else:
            OOF[SL, j] = bst.predict(D_SL); OOF[VA, j] = bst.predict(D_VA)
        done.add(key); ensure_res(); np.save(P_OOF, OOF)
        json.dump(sorted(done), open(P_PROG, 'w'))
    print(f'  [{j+1:>2}/{C}] {LAB40[j]:<28} n+={npos:>6,}  {(time.time()-t0)/60:6.1f} ph',
          flush=True)

# test KHONG duoc du doan — ky luat "test chua he cham". Dien hang so prevalence cua LAT KHOP.
_prev = YM[TF_].mean(0)
OOF[TEST] = _prev[None, :].astype('float32')
ensure_res(); np.save(P_OOF, OOF)
assert np.isfinite(OOF).all(), f'con {int((~np.isfinite(OOF)).sum()):,} o chua duoc dien'
print(f'\nxong | {(time.time()-t0)/60:.1f} phut | ghi {P_OOF}')
print(f'  hang test ({TEST.sum():,}) dien HANG SO prevalence cua lat khop — khong du doan, '
      'khong doc nhan test.')

XGB tren cpu | 300 vong | depth 6 | 40 nhan x (5+1) = 240 lan fit
  [ 1/40] Adenopathy                   n+= 2,920     1.8 ph
  [ 2/40] Atelectasis                  n+=55,489     3.5 ph
  [ 3/40] Azygos Lobe                  n+=   176     5.7 ph
  [ 4/40] Calcification of the Aorta   n+= 3,670     7.5 ph
  [ 5/40] Cardiomegaly                 n+=63,420     9.3 ph
  [ 6/40] Clavicle Fracture            n+=   147    11.4 ph
  [ 7/40] Consolidation                n+=13,182    13.2 ph
  [ 8/40] Edema                        n+=31,602    14.9 ph
  [ 9/40] Emphysema                    n+= 3,124    16.8 ph
  [10/40] Enlarged Cardiomediastinum   n+=25,136    18.4 ph
  [11/40] Fibrosis                     n+=   981    20.3 ph
  [12/40] Fissure                      n+= 2,295    22.0 ph
  [13/40] Fracture                     n+= 9,900    23.8 ph
  [14/40] Granuloma                    n+= 2,508    25.6 ph
  [15/40] Hernia                       n+= 3,467    27.4 ph
  [16/40] Hydropneumothorax       

## C3 · Cổng — tái lập `46`, và chứng minh **không rò rỉ**

Ba cổng, nêu ngưỡng trước:

```
1  du doan VA phai TAI LAP so cua 46          |mAP(VA) - 0.1241| < 0.005
2  OOF tren lat khop KHONG duoc cao hon VA     mAP(OOF) - mAP(VA) < 0.02
      cao hon nhieu = model da thay nhan cua chinh hang do
3  hoan vi nhan -> mAP phai sup ve prevalence  (doi chung am)
```

Cổng 2 là cổng thật. Một đặc trưng rò rỉ sẽ cho `mAP(OOF)` gần `1.0`, không phải `~0.12`.

In [5]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

AP_VA  = ap_cols(YM[VA],  OOF[VA]);  M_VA  = float(np.nanmean(AP_VA))
AP_OOF = ap_cols(YM[TF_], OOF[TF_]); M_OOF = float(np.nanmean(AP_OOF))
AP_SL  = ap_cols(YM[SL],  OOF[SL]);  M_SL  = float(np.nanmean(AP_SL))
print(f'{"lat":<12} {"mAP":>8}  ' + ' '.join(f'{t:>8}' for t in ('head','medium','tail')))
for nm, ap, m in (('val', AP_VA, M_VA), ('OOF khop', AP_OOF, M_OOF), ('chon', AP_SL, M_SL)):
    print(f'{nm:<12} {m:>8.4f}  ' + ' '.join(f'{v:>8.4f}' for v in tiers(ap).values()))

_d1 = abs(M_VA - MOC46_VA)
print(f'\ncong 1  mAP(val) {M_VA:.4f} vs 46 {MOC46_VA:.4f}  ->  lech {_d1:.4f}')
assert _d1 < 0.005, (f'KHONG tai lap duoc so cua 46 (lech {_d1:.4f}). Dac trung hoac lat cat '
                     'da khac -> DUNG dung file nay lam dac trung cho 52b.')
_d2 = M_OOF - M_VA
print(f'cong 2  mAP(OOF khop) - mAP(val) = {_d2:+.4f}   (CHAN DOAN, khong phai cong chinh)')
print('        Am nhe la BINH THUONG: model OOF chi hoc tren 80% lat khop, con model cho `val`')
print('        hoc tren 100%. Cong CHINH chong ro ri la cong 1 o C0 (cau truc, khong nguong) —')
print('        dry-run da cho thay nguong thong ke o day KHONG bat duoc fold chia theo hang.')
assert _d2 < 0.05, (f'OOF cao hon val {_d2:+.4f} -> ro ri THO. Kiem lai C0.')
_rs3 = np.random.default_rng(3); _pp = _rs3.permutation(int(VA.sum()))
_m3 = float(np.nanmean(ap_cols(YM[VA], OOF[VA][_pp])))
_pv = float(np.nanmean(YM[VA].mean(0)))
print(f'cong 3  hoan vi nhan -> mAP {_m3:.4f} (prevalence TB {_pv:.4f})')
assert abs(_m3 - _pv) < 0.01, f'doi chung am khong sup ve prevalence: {_m3:.4f} vs {_pv:.4f}'
print('\nba cong DAT — `xgb_oof.npy` dung duoc lam dac trung cho 52b.')

_t = pd.DataFrame(dict(nhan=LAB40, n_pos_khop=YM[TF_].sum(0).astype(int),
                       ap_val=AP_VA, ap_oof=AP_OOF)).sort_values('ap_val', ascending=False)
print(f'\n8 nhan XGB doc TOT nhat / 8 nhan kem nhat (mAP val):')
print(_t.head(8).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(_t.tail(8).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
ensure_res(); _t.to_csv(os.path.join(RES52, 'xgb_per_label.csv'), index=False)
print(f'\n=> 52b nap: np.load("{P_OOF}")  [{len(ORDER):,} x {C}]')
print('   Trong 52b, 40 cot nay di qua CHINH `ColTok` ma `38*` dung — chi doi DAC TRUNG DAU VAO,')
print('   khong doi kien truc. Nen `G_ehr(xgb) - G_ehr(col)` doc duoc thang.')

lat               mAP      head   medium     tail
val            0.1241    0.3260   0.0414   0.0059
OOF khop       0.1219    0.3205   0.0413   0.0045
chon           0.1237    0.3185   0.0443   0.0090

cong 1  mAP(val) 0.1241 vs 46 0.1241  ->  lech 0.0000
cong 2  mAP(OOF khop) - mAP(val) = -0.0022   (CHAN DOAN, khong phai cong chinh)
        Am nhe la BINH THUONG: model OOF chi hoc tren 80% lat khop, con model cho `val`
        hoc tren 100%. Cong CHINH chong ro ri la cong 1 o C0 (cau truc, khong nguong) —
        dry-run da cho thay nguong thong ke o day KHONG bat duoc fold chia theo hang.
cong 3  hoan vi nhan -> mAP 0.0619 (prevalence TB 0.0617)

ba cong DAT — `xgb_oof.npy` dung duoc lam dac trung cho 52b.

8 nhan XGB doc TOT nhat / 8 nhan kem nhat (mAP val):
            nhan  n_pos_khop  ap_val  ap_oof
 Support Devices       73311  0.7261  0.7414
Pleural Effusion       56451  0.5768  0.5639
    Cardiomegaly       63420  0.4918  0.4807
    Lung Opacity       65851  0.4398  0.4341
    